# 分子动力学演示：Lennard-Jones 粒子模拟

由灵光编写，存放在 GitHub 仓库 `colab-md`。

这个笔记本做一件小事：把一团粒子放进盒子里，让它们按 Lennard-Jones 势互相吸引、排斥，跑一段分子动力学（MD），看看能量怎么变化。

如果 Colab 分到了 GPU，计算就在显卡上跑；没分到也能用 CPU 跑完，只是慢一点。

In [ ]:
import torch
print("PyTorch 版本:", torch.__version__)
print("CUDA 可用:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU 型号:", torch.cuda.get_device_name(0))
device = "cuda" if torch.cuda.is_available() else "cpu"
print("本次使用设备:", device)

## 模拟设置

- N 个粒子，随机初始位置，避免重叠
- Lennard-Jones 势：$V(r) = 4\varepsilon\left[(\sigma/r)^{12} - (\sigma/r)^{6}\right]$
- 速度 Verlet 积分，周期性边界条件
- 单位约定：σ = ε = m = 1

In [ ]:
import matplotlib.pyplot as plt

torch.manual_seed(42)

N = 500          # 粒子数
L = 12.0         # 盒子边长
dt = 0.002       # 时间步长
steps = 2000     # 总步数
rcut = 2.5       # 截断半径

# 初始位置：网格摆放加少量扰动，避免粒子重叠
grid_n = int(N ** (1/3)) + 1
spacing = L / grid_n
pos = []
for i in range(grid_n):
    for j in range(grid_n):
        for k in range(grid_n):
            if len(pos) < N:
                p = torch.tensor([i, j, k], dtype=torch.float32) * spacing + 0.5 * spacing
                pos.append(p)
pos = torch.stack(pos).to(device)
pos += 0.05 * spacing * (torch.rand_like(pos) - 0.5)

# 初始速度：温度对应的随机速度，去掉整体动量
vel = torch.randn(N, 3, device=device) * 0.5
vel -= vel.mean(dim=0, keepdim=True)

def lj_forces(pos, L, rcut):
    """计算 LJ 力与势能（周期性边界，截断半径）"""
    d = pos.unsqueeze(1) - pos.unsqueeze(0)          # (N, N, 3)
    d = d - L * torch.round(d / L)                   # 最小镜像
    r2 = (d ** 2).sum(-1)                            # (N, N)
    mask = (r2 < rcut ** 2) & (r2 > 1e-8)
    r2_safe = r2.clamp(min=1e-8)
    inv_r2 = 1.0 / r2_safe
    inv_r6 = inv_r2 ** 3
    # 力的大小 f/r = 24 * eps * (2*inv_r12 - inv_r6) / r2
    f_over_r = 24.0 * (2.0 * inv_r6 ** 2 - inv_r6) * inv_r2
    f_over_r = torch.where(mask, f_over_r, torch.zeros_like(f_over_r))
    forces = (f_over_r.unsqueeze(-1) * d).sum(dim=1)  # (N, 3)
    # 势能（截断处不修正，演示用）
    pe = torch.where(mask, 4.0 * (inv_r6 ** 2 - inv_r6), torch.zeros_like(inv_r6))
    pe = pe.sum() / 2.0
    return forces, pe

forces, pe = lj_forces(pos, L, rcut)
pe_hist, ke_hist, t_hist = [], [], []

for step in range(steps):
    vel += forces * dt / 2.0
    pos = (pos + vel * dt) % L
    forces, pe = lj_forces(pos, L, rcut)
    vel += forces * dt / 2.0
    if step % 20 == 0:
        ke = 0.5 * (vel ** 2).sum().item()
        pe_hist.append(pe.item())
        ke_hist.append(ke)
        t_hist.append(step * dt)

print(f"模拟完成：{N} 个粒子，{steps} 步，设备 {device}")
print(f"末态动能 {ke_hist[-1]:.2f}，势能 {pe_hist[-1]:.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(t_hist, ke_hist, label="Kinetic")
axes[0].plot(t_hist, pe_hist, label="Potential")
axes[0].plot(t_hist, [k + p for k, p in zip(ke_hist, pe_hist)], label="Total", alpha=0.7)
axes[0].set_xlabel("Time")
axes[0].set_ylabel("Energy")
axes[0].set_title("Energy vs Time")
axes[0].legend()

pos_cpu = pos.cpu().numpy()
axes[1].scatter(pos_cpu[:, 0], pos_cpu[:, 1], s=8, alpha=0.7)
axes[1].set_xlim(0, L)
axes[1].set_ylim(0, L)
axes[1].set_title("Final positions (xy projection)")
axes[1].set_aspect("equal")

plt.tight_layout()
plt.show()

## 看什么

- **总能量**应该基本走平——这是数值积分守恒性的直观检验；
- **动能和势能**此消彼长：粒子从随机分布往「抱团」演化，势能下降、动能上升，最后在一个温度附近震荡；
- 右图是末态粒子在 xy 平面的投影，能看到它们聚成了一小团一小的「液滴」。

想玩大一点：把 `N` 调到 4000、`steps` 调到 10000，GPU 上也就多跑一会儿。